# Module 03: AI in Healthcare - Clinical Symptom Checker & Triage Tool
**Author:** Sergio Sanchez  
**Course:** 6271-ITAI-2372-Artificial Intel Applications  
**Topic:** Natural Language Processing (NLP) & Rule-Based Clinical Triage  

---

## 1. Executive Summary & Clinical Problem
In outpatient and emergency settings, patient intake bottlenecks often delay urgent care access. This project demonstrates a prototype **AI-Powered Symptom Checker and Triage Engine** that ingests unstructured patient descriptions, processes natural language into standardized tokens, and matches clinical presentations against an annotated medical knowledge base.

### The Objective
1. Standardize freeform patient language using standard NLP methods (tokenization, stop-word removal, lemmatization).
2. Quantify symptom similarity against defined emergency, urgent, primary care, and self-care profiles.
3. Provide automated preliminary triage prioritization while preserving human-in-the-loop clinical escalation.

## 2. Environment Setup & Lexical Resources
The preprocessing pipeline relies on the Natural Language Toolkit (`nltk`). We download the required corpora:
- `punkt` / `punkt_tab`: Tokenizer models for boundary splitting.
- `stopwords`: Lexical stop-words for noise reduction.
- `wordnet`: Lexical database for morphological lemmatization.

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK datasets quietly
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("NLTK linguistic resources initialized successfully.")

NLTK linguistic resources initialized successfully.


## 3. Data Architecture: The Annotated Clinical Knowledge Base
To adhere to the Data-Centric AI principles covered in Module 02, our baseline knowledge base stores structured symptom feature sets alongside:
- **Preliminary Condition:** The potential underlying pathology.
- **Triage Level:** Categorized by acuity (`EMERGENCY`, `URGENT CARE`, `PRIMARY CARE`, `SELF-CARE`).
- **Clinical Protocol:** The immediate downstream action.

In [2]:
# Format: "key symptom terms": ("Preliminary Diagnosis", "Triage Level", "Recommended Protocol")
SYMPTOM_KNOWLEDGE_BASE = {
    "chest pain shortness breath sweating pressure": (
        "Possible Acute Coronary Syndrome (Myocardial Infarction)",
        "EMERGENCY",
        "Activate emergency medical services (911) immediately. Do not drive to hospital."
    ),
    "severe headache stiff neck high fever confusion": (
        "Possible Meningitis / Neurological Infection",
        "EMERGENCY",
        "Seek emergency room clinical evaluation immediately."
    ),
    "sudden numbness weakness facial droop speech difficulty": (
        "Possible Cerebrovascular Accident (Stroke)",
        "EMERGENCY",
        "Activate 911 immediately. Note exact onset time for thrombolytic window."
    ),
    "persistent high fever productive cough chills fatigue": (
        "Lower Respiratory Infection / Suspected Pneumonia",
        "URGENT CARE",
        "Obtain a same-day urgent care evaluation for chest auscultation/imaging."
    ),
    "severe abdominal pain nausea vomiting right lower quadrant": (
        "Suspected Acute Appendicitis",
        "URGENT CARE",
        "Report to an urgent care or emergency surgical intake center."
    ),
    "mild headache low fever runny nose sore throat": (
        "Viral Upper Respiratory Infection (Common Cold / Mild Flu)",
        "PRIMARY CARE",
        "Hydrate, rest, and follow up with a primary care physician if symptoms persist > 5 days."
    ),
    "itchy watery eyes sneezing clear nasal discharge": (
        "Seasonal Allergic Rhinitis",
        "SELF-CARE",
        "Consider over-the-counter antihistamines and allergen avoidance."
    ),
    "mild localized joint pain stiffness morning": (
        "Musculoskeletal Strain / Early Osteoarthritis",
        "SELF-CARE",
        "Engage in low-impact mobility, rest, and schedule routine clinical review."
    )
}

print(f"Loaded {len(SYMPTOM_KNOWLEDGE_BASE)} clinical diagnostic profiles.")

Loaded 8 clinical diagnostic profiles.


## 4. Text Preprocessing Pipeline
Unstructured freeform inputs contain grammatical noise, varying capitalization, and inflectional endings. The preprocessing function standardizes user statements:
1. **Lowercasing & Tokenization:** Deconstructs raw sentences into discrete lowercase tokens.
2. **Stop-Word & Non-Alpha Stripping:** Discards functional words (e.g., "the", "and", "is") and punctuation.
3. **Lemmatization:** Reduces morphological inflections to their dictionary lemma (e.g., "vomiting" $\rightarrow$ "vomit", "fevers" $\rightarrow$ "fever").

In [3]:
def preprocess_text(text: str) -> list[str]:
    """
    Transforms raw patient symptom narrative into clean, lemmatized tokens.
    """
    # 1. Tokenize input text
    tokens = word_tokenize(text.lower())
    
    # 2. Filter non-alphabetic elements and common stop-words
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    
    # 3. Lemmatize tokens to root grammatical base
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(w) for w in filtered_tokens]
    
    return lemmatized_tokens

# Verification test
test_phrase = "The patient was experiencing severe fevers and breathing difficulties."
print("Sample Input :", test_phrase)
print("Normalized   :", preprocess_text(test_phrase))

Sample Input : The patient was experiencing severe fevers and breathing difficulties.
Normalized   : ['patient', 'experiencing', 'severe', 'fever', 'breathing', 'difficulty']


## 5. Scoring & Triage Inference Logic
The engine iterates over the knowledge base, calculating the intersection between the processed patient tokens and reference symptom signatures. 
- The profile yielding the highest keyword overlap count is selected.
- If no significant match is found, the system fails safely to an `Indeterminate` state requiring routine medical examination.

In [4]:
def analyze_symptoms(user_text: str) -> dict:
    """
    Performs keyword intersection matching between input tokens and reference symptom maps.
    """
    tokens = preprocess_text(user_text)
    token_set = set(tokens)
    
    best_condition = None
    best_triage = None
    best_action = None
    max_overlap = 0

    for symptom_signature, (diagnosis, triage, protocol) in SYMPTOM_KNOWLEDGE_BASE.items():
        signature_tokens = set(symptom_signature.split())
        overlap_score = len(token_set.intersection(signature_tokens))
        
        if overlap_score > max_overlap:
            max_overlap = overlap_score
            best_condition = diagnosis
            best_triage = triage
            best_action = protocol

    # Fallback safe handling
    if max_overlap >= 1:
        return {
            "processed_tokens": tokens,
            "preliminary_assessment": best_condition,
            "triage_priority": best_triage,
            "action_protocol": best_action,
            "confidence_score": max_overlap
        }
    else:
        return {
            "processed_tokens": tokens,
            "preliminary_assessment": "Indeterminate / Unmatched Symptom Profile",
            "triage_priority": "PRIMARY CARE",
            "action_protocol": "Symptoms require manual evaluation. Schedule an appointment with a licensed physician.",
            "confidence_score": 0
        }

print("Inference engine defined and ready.")

Inference engine defined and ready.


## 6. Verification Test Suite
We evaluate the pipeline across three distinct clinical scenarios representing different urgency tiers:
1. **Critical Cardiovascular Presentation (Emergency)**
2. **Pediatric Viral Presentation (Primary Care)**
3. **Acute Neurological Deficit (Emergency)**

In [5]:
test_cases = [
    "I have sudden crushing chest pain, sweating, and difficulty breathing.",
    "My child has a mild headache, a sore throat, and a runny nose.",
    "My father has sudden numbness in his arm and weakness on his face."
]

print("=" * 80)
print("CLINICAL SYMPTOM INTAKE & TRIAGE EXECUTION RESULTS")
print("=" * 80)

for idx, statement in enumerate(test_cases, start=1):
    result = analyze_symptoms(statement)
    print(f"\n[Case {idx}] Patient Statement: \"{statement}\"")
    print(f"  • Extracted Tokens    : {result['processed_tokens']}")
    print(f"  • Preliminary Assess  : {result['preliminary_assessment']}")
    print(f"  • Triage Priority     : [{result['triage_priority']}]")
    print(f"  • Recommended Protocol: {result['action_protocol']}")

print("\n" + "=" * 80)

CLINICAL SYMPTOM INTAKE & TRIAGE EXECUTION RESULTS

[Case 1] Patient Statement: "I have sudden crushing chest pain, sweating, and difficulty breathing."
  • Extracted Tokens    : ['sudden', 'crushing', 'chest', 'pain', 'sweating', 'difficulty', 'breathing']
  • Preliminary Assess  : Possible Acute Coronary Syndrome (Myocardial Infarction)
  • Triage Priority     : [EMERGENCY]
  • Recommended Protocol: Activate emergency medical services (911) immediately. Do not drive to hospital.

[Case 2] Patient Statement: "My child has a mild headache, a sore throat, and a runny nose."
  • Extracted Tokens    : ['child', 'mild', 'headache', 'sore', 'throat', 'runny', 'nose']
  • Preliminary Assess  : Viral Upper Respiratory Infection (Common Cold / Mild Flu)
  • Triage Priority     : [PRIMARY CARE]
  • Recommended Protocol: Hydrate, rest, and follow up with a primary care physician if symptoms persist > 5 days.

[Case 3] Patient Statement: "My father has sudden numbness in his arm and weakness on h

## 7. Responsible AI, Data-Centric Governance, & Clinical Disclaimers

### Ethical Constraints & Human-in-the-Loop Safeguards
1. **Model Drift & Semantic Limitations:** Keyword overlap cannot capture subtle physiological nuances, clinical modifiers ("no chest pain" vs. "severe chest pain"), or non-verbal distress indicators.
2. **Clinical Verification:** AI triage must serve as an intake accelerator, never a final diagnostic decision-maker. Critical edge cases must route directly to clinical triage nurses.
3. **Data Privacy (HIPAA):** Production healthcare agents require de-identification pipelines to ensure protected health information (PHI) is purged before processing.

---
### Clinical Prototype Notice
*This software is an educational prototype developed for course demonstration purposes only. It is not approved medical diagnostic software. Always consult a certified healthcare professional for medical emergencies and clinical evaluations.*